In [13]:
# imports
import os
import io
import math
from datetime import datetime, timezone

import requests
import pandas as pd

from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

openai = OpenAI()

print("Imports loaded.")
print("Notebook time (UTC):", datetime.now(timezone.utc).isoformat(timespec="seconds"))

Imports loaded.
Notebook time (UTC): 2026-03-06T06:59:21+00:00


In [14]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


In [16]:
# Scrape daily stock market quotes
# Data source: Stooq public CSV

WATCHLIST = [
    "^spx",
    "^ndq",
    "^dji",
    "spy.us",
    "qqq.us",
    "aapl.us",
    "msft.us",
]

LOOKBACK_ROWS = 90


def fetch_stooq_daily(symbol: str) -> pd.DataFrame:
    url = f"https://stooq.com/q/d/l/?s={symbol}&i=d"
    headers = {
        "User-Agent": "Mozilla/5.0 (MarketReportNotebook/1.0)",
        "Accept": "text/csv,*/*",
    }

    r = requests.get(url, headers=headers, timeout=30)
    r.raise_for_status()

    text = r.text.strip()

    if not text or "Date" not in text.splitlines()[0]:
        first = text.splitlines()[0] if text else "EMPTY"
        raise ValueError(f"Unexpected response for {symbol}. First line: {first}")

    df = pd.read_csv(io.StringIO(text))
    if df.empty:
        raise ValueError(f"No data for symbol {symbol}")

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)


    for c in ["Open", "High", "Low", "Close", "Volume"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    df["Symbol"] = symbol
    return df


def pct_change_over_days(close: pd.Series, days: int) -> float:
    close = close.dropna()
    if len(close) < days + 1:
        return float("nan")
    return float(close.iloc[-1] / close.iloc[-(days + 1)] - 1.0)


def annualized_volatility(close: pd.Series, window: int = 20) -> float:
    close = close.dropna()
    if len(close) < window + 1:
        return float("nan")

    # log return
    log_ret = (close / close.shift(1)).apply(
        lambda x: math.log(x) if pd.notna(x) and x > 0 else float("nan")
    )
    vol = log_ret.dropna().tail(window).std(ddof=1) * math.sqrt(252)
    return float(vol)


def build_market_summary(watchlist: list[str]) -> tuple[pd.DataFrame, list[str]]:
    frames: dict[str, pd.DataFrame] = {}
    errors: list[str] = []

    for sym in watchlist:
        try:
            df = fetch_stooq_daily(sym)
            frames[sym] = df.tail(LOOKBACK_ROWS).reset_index(drop=True)
        except Exception as e:
            errors.append(f"{sym}: {type(e).__name__}: {e}")

    rows = []
    for sym, df in frames.items():
        close = df["Close"].dropna()
        if len(close) < 2:
            continue

        last_close = float(close.iloc[-1])
        prev_close = float(close.iloc[-2])
        chg = last_close - prev_close
        chg_pct = (last_close / prev_close - 1.0) if prev_close else float("nan")

        rows.append(
            {
                "Symbol": sym,
                "Date": df["Date"].iloc[-1].date().isoformat(),
                "Close": last_close,
                "Chg": chg,
                "Chg%": chg_pct,
                "5D%": pct_change_over_days(close, 5),
                "20D%": pct_change_over_days(close, 20),
                "Vol(20D)": annualized_volatility(close, 20),
                "Volume": float(df["Volume"].iloc[-1]) if "Volume" in df.columns else float("nan"),
            }
        )

    summary = pd.DataFrame(rows)
    if not summary.empty:
        summary = summary.sort_values("Chg%", ascending=False).reset_index(drop=True)

    return summary, errors


def to_markdown_table(summary: pd.DataFrame) -> str:
    if summary.empty:
        return "(no data)"

    disp = summary.copy()

    disp["Close"] = disp["Close"].map(lambda x: f"{x:,.2f}" if pd.notna(x) else "")
    disp["Chg"] = disp["Chg"].map(lambda x: f"{x:+,.2f}" if pd.notna(x) else "")
    disp["Chg%"] = disp["Chg%"].map(lambda x: f"{x:+.2%}" if pd.notna(x) else "")
    disp["5D%"] = disp["5D%"].map(lambda x: f"{x:+.2%}" if pd.notna(x) else "")
    disp["20D%"] = disp["20D%"].map(lambda x: f"{x:+.2%}" if pd.notna(x) else "")
    disp["Vol(20D)"] = disp["Vol(20D)"].map(lambda x: f"{x:.1%}" if pd.notna(x) else "")
    disp["Volume"] = pd.to_numeric(disp["Volume"], errors="coerce").map(
        lambda x: f"{x:,.0f}" if pd.notna(x) else ""
    )


    headers = list(disp.columns)
    lines = ["| " + " | ".join(headers) + " |", "|" + "|".join(["---"] * len(headers)) + "|"]
    for _, row in disp.iterrows():
        lines.append("| " + " | ".join(str(row[h]) for h in headers) + " |")
    return "\n".join(lines)


market_summary, market_errors = build_market_summary(WATCHLIST)

if market_errors:
    display(Markdown("**fail to retrieve some symbols:**\n\n- " + "\n- ".join(market_errors)))

if market_summary.empty:
    display(Markdown("**no data retrieved**"))
else:
    display(Markdown("## Market Summary"))
    display(market_summary)

market_table_md = to_markdown_table(market_summary)
print("\nMarkdown table for prompt is ready. Rows:", len(market_summary))

## Market Summary

,Symbol,Date,Close,Chg,Chg%,5D%,20D%,Vol(20D),Volume
0,msft.us,2026-03-05,410.68,5.68,0.014025,0.022304,-0.008474,0.320558,3.900132e+07
1,^ndq,2026-03-05,22748.99,-58.49,-0.002565,-0.005656,-0.006793,0.175575,6.752916e+09
2,qqq.us,2026-03-05,608.93,-1.79,-0.002931,-0.000509,0.005250,0.174020,8.960241e+07
3,spy.us,2026-03-05,681.31,-3.82,-0.005576,-0.011591,-0.007112,0.132232,1.066065e+08
4,^spx,2026-03-05,6830.71,-38.79,-0.005647,-0.011312,-0.007557,0.134405,3.695803e+09
5,aapl.us,2026-03-05,260.25,-2.20,-0.008383,-0.046529,-0.058736,0.292676,4.965863e+07
6,^dji,2026-03-05,47954.74,-784.67,-0.016099,-0.031202,-0.031243,0.153537,4.711238e+08



Markdown table for prompt is ready. Rows: 7


In [17]:
# Step 1: Create your prompts

system_prompt = """
You are a professional stock/macro market analyst and investment assistant. You only analyze the market table provided, do not make up data.
Output requirements:
- First, give 5-10 bullet points
- Then, give a 150-250 word summary
- Finally, provide 3 actionable points of attention (e.g., risks, key levels, macro events/earnings reports, etc.)
"""

user_prompt = f"""
Below is a table of today's market data (may include indices, ETFs, and individual stocks). Please summarize today's market performance based on the table:
{market_table_md}
- Please indicate if the table is empty or lacks sufficient data.
- Pay close attention to stocks with the largest price fluctuations and higher volatility.
"""

# Step 2: Make the messages list
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]

# Step 3: Call OpenAI
response = openai.chat.completions.create(
    model="gpt-5-nano",
    messages=messages,
)

# Step 4: print the result
text = response.choices[0].message.content
display(Markdown(text))

- The table is populated with data for MSFT, Nasdaq components (^NDQ, QQQ), SPY, SPX (^SPX), AAPL, and DJIA (^DJI) dated 2026-03-05.
- Major index performance: Dow Jones Industrial Average (-1.61%) led downside; S&P 500 (-0.56%) and SPY (-0.56%) also lower; Nasdaq composites were modestly negative (~-0.26% to -0.29%).
- MSFT was the standout on the day, closing up 1.40% (+5.68 points) to 410.68, with unusually strong volume relative to the 20-day average (+32.1%).
- The tech/large-cap leadership was mixed: QQQ (-0.29%) and ^NDQ (-0.26%) weakened, signaling modest breadth softness despite MSFT’s gain.
- AAPL fell 0.84% to 260.25, with longer-term momentum negative (5D% -4.65%, 20D% -5.87%) and a high relative volume (+29.3%).
- Momentum snapshots: MSFT shows positive 5D% (+2.23%) but negative 20D% (-0.85%), suggesting a short-term bounce within a softer medium-term trend.
- Volume context: elevated trading activity on MSFT (Vol(20D) +32.1%) and AAPL (Vol(20D) +29.3%) indicates notable participation around the leadership names.
- The largest absolute move among components was DJIA’s drop of 784.67 points (-1.61%), highlighting broad downside pressure even as MSFT advanced.
- Data completeness: The table is not empty and contains sufficient data for a concise daily-read summary.

150-250 word summary:
Today’s session yielded a mixed-to-soft tone for U.S. equities. The Dow underperformed, closing down 1.61% (−784.67 points), signaling risk-off pressure at the index level. The S&P 500 and SPY declined about 0.56%, while the Nasdaq indices were modestly negative (around −0.26% to −0.29%). In contrast to the broad weakness, Microsoft stood out on the day, rising 1.40% to 410.68 with a robust volume surge (20D volume +32.1%), suggesting solid intraday conviction and potential leadership rotation within large caps. Apple, however, lagged with an −0.84% close to 260.25; its 5D and 20D momentum readings remain deeply negative (−4.65% and −5.87%), and it also showed strong relative volume (+29.3%), indicating active trading but continued downside momentum.

Overall, breadth was soft despite MSFT’s strength, with the QQQ and ^NDQ drifting lower while DJIA bore the heaviest losses. The combination of elevated volume on MSFT and AAPL against a softer broader market points to a possible rotation or consolidation phase rather than a broad risk-on move. The data suggest near-term leadership tentatively shifting toward select mega-cap tech despite an overall negative daily tone.

3 actionable points of attention:
- Leadership rotation risk: MSFT led higher on substantial volume while the Dow declined meaningfully. Monitor if MSFT can sustain above today’s close (410.68) and whether other mega-cap tech names begin to participate; failure could signal a broader tech-led pullback.
- Key level references and risk signals: Watch the close levels around SPY 681.31, SPX 6,830.71, DJIA 47,954.74, and QQQ 608.93 as near-term reference points. A break below these levels could signal renewed downside pressure; a move back toward or above these levels may indicate stabilization.
- Volume and momentum cues: Elevated Vol(20D) for MSFT (+32.1%) and AAPL (+29.3%) implies conviction behind moves. If volume sustains with further price action, it may precede a continuation; if volume drops while prices drift, expect potential reversals. Also note the negative medium-term momentum for AAPL (20D −5.87%), suggesting relative weakness that warrants monitoring for any deterioration or reversal catalysts.